# Tutorial 9: Audio Generation with IndexTTS

<p align="right" style="margin-right: 8px;">
    <a target="_blank" href="https://colab.research.google.com/github/idiap/sdialog/blob/main/tutorials/01_audio/9.IndexTTS.ipynb">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
    </a>
</p>

## Getting started

### Environment Setup

Let's first check if our environment is all set up.

In [ ]:
# Setup the environment depending on weather we are running in Google Colab or Jupyter Notebook
import os
from IPython import get_ipython
from IPython.display import display

if "google.colab" in str(get_ipython()):
    print("Running on CoLab")
    !sudo apt-get install sox ffmpeg
    !git clone https://github.com/idiap/sdialog.git
    %cd sdialog
    %pip install -e .[audio]
    %pip install git+https://github.com/index-tts/index-tts.git
    %hf download "sdialog/voices-jsalt" --repo-type dataset
    %hf download "facebook/w2v-bert-2.0" --repo-type model
    # %hf download "sdialog/voices-libritts" --repo-type dataset
    %cd ..
else:
    print("Running in Jupyter Notebook")

### Load the example dialogue

In order to run the next steps in a fast manner, we will start from an existing dialog generated using previous tutorials:

In [ ]:
from sdialog import Dialog

path_dialog = "../../tests/data/demo_dialog_doctor_patient.json"

if not os.path.exists(path_dialog):
    !wget https://raw.githubusercontent.com/idiap/sdialog/refs/heads/main/tests/data/demo_dialog_doctor_patient.json
    path_dialog = "./demo_dialog_doctor_patient.json"

original_dialog = Dialog.from_file(path_dialog)
original_dialog.print()
original_dialog.turns = original_dialog.turns[0:4]
print(len(original_dialog.turns))

In [ ]:
INDEXTTS_VERSION = "1.5"
# INDEXTTS_VERSION = "2"

In [ ]:
from sdialog.audio.normalizers import LowercaseNormalizer, WhisperNormalizer, ReplaceCommaWithDotNormalizer
normalizers = [ReplaceCommaWithDotNormalizer(), LowercaseNormalizer(), WhisperNormalizer()]

In [ ]:
from sdialog.audio.tts import IndexTTS
from sdialog.audio.voice_database import HuggingfaceVoiceDatabase

if INDEXTTS_VERSION == "1.5":
    tts_engine = IndexTTS(model_dir="checkpoints-15", cfg_path="checkpoints-15/config.yaml", version="1", text_normalizers=normalizers)
else:
    tts_engine = IndexTTS(model_dir="checkpoints", cfg_path="checkpoints/config.yaml", version="2", text_normalizers=normalizers)

In [ ]:
hf_repo = "sdialog/voices-celebrities"
# hf_repo = "sdialog/voices-jsalt"
# hf_repo = "sdialog/voices-libritts"

In [ ]:
voice_database = HuggingfaceVoiceDatabase(hf_repo)

## Setup stage: Audio Dialog and Audio Pipeline

In [ ]:
from sdialog.audio.dialog import AudioDialog
dialog: AudioDialog = AudioDialog.from_dialog(original_dialog)

In [ ]:
from sdialog.audio.utils import SourceVolume, SourceType
from sdialog.audio.room import SpeakerSide, Role, RoomPosition
from sdialog.audio.jsalt import MedicalRoomGenerator, RoomRole

In [ ]:
room = MedicalRoomGenerator().generate(args={"room_type": RoomRole.TREATMENT})
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_1, furniture_name="desk", max_distance=1.0, side=SpeakerSide.FRONT)
room.place_speaker_around_furniture(speaker_name=Role.SPEAKER_2, furniture_name="desk", max_distance=1.0, side=SpeakerSide.BACK)

In [ ]:
new_audio_dialog = original_dialog.to_audio(
    tts_engine=tts_engine,
    voice_database=voice_database,
    perform_room_acoustics=True,
    room=room,
    dir_audio="./outputs_indextts",
    dialog_dir_name="demo_indextts-15" if INDEXTTS_VERSION == "1.5" else "demo_indextts",
    room_name=f"base_room",
    background_effect="white_noise",
    foreground_effect="ac_noise_minimal",
    foreground_effect_position=RoomPosition.TOP_RIGHT,
    source_volumes={
        SourceType.ROOM: SourceVolume.HIGH,
        SourceType.BACKGROUND: SourceVolume.VERY_LOW
    },
    kwargs_pyroom={
        "ray_tracing": True,
        "air_absorption": True
    },
    override_tts_audio=True,
    verbose=True,
    voices={
        Role.SPEAKER_1: ("donald_trump","english"),
        Role.SPEAKER_2: ("bill_gates","english"),
    } if hf_repo == "sdialog/voices-celebrities" else None
)

In [ ]:
new_audio_dialog.display()